# Preprocessing: lemmatized descriptions

This notebook runs the preprocessing for variant B of the classical models and checks its output.
Variant B replaces each description with the lemmas of its content words: spaCy's French pipeline
(`fr_core_news_md`, pinned in `pyproject.toml`) lemmatizes the text and drops stop words,
punctuation and whitespace. Variant A and CamemBERTav2 need no stored preprocessing: TF-IDF
lowercases and tokenizes inside its own pipeline, and the transformer reads the original text.

The code lives in `geneva_permits.preprocess`, this notebook only calls it. Examples and statistics
use training rows only, so no validation or test description is displayed.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

from geneva_permits.config import PROJECT_ROOT
from geneva_permits.preprocess import is_kept, load_nlp, load_texts, write_texts

pd.set_option("display.max_colwidth", None)

nlp = load_nlp()
print("Model:", f"{nlp.meta['lang']}_{nlp.meta['name']}", nlp.meta["version"])
print("Components:", nlp.pipe_names)

Model: fr_core_news_md 3.8.0
Components: ['tok2vec', 'morphologizer', 'attribute_ruler', 'lemmatizer']


## 1. Build and store

**Question:** Which records are preprocessed, and with which versions?

**Method:** `write_texts` lemmatizes every retained record outside the unlabelled pool and stores
the original text, the lemmas and the spaCy and model versions in `data/processed/texts.parquet`.
`load_texts` reads the file back and stops if those versions differ from the installed ones.

In [2]:
path, rows = write_texts()
texts = load_texts()
print(f"Wrote {rows:,} rows to {path.relative_to(PROJECT_ROOT)} with {texts.attrs}")
display(texts["partition"].value_counts())

train = texts.loc[texts["partition"].eq("train_original")]

Wrote 41,567 rows to data/processed/texts.parquet with {'spacy': '3.8.16', 'fr_core_news_md': '3.8.0'}


partition
train_original     30782
test_recent         6671
validation          2794
test_historical     1320
Name: count, dtype: int64[pyarrow]

**Finding:** The file holds 41,567 rows: 30,782 training, 2,794 validation, 1,320 historical-test
and 6,671 recent-test records, matching the retained partitions of notebook 01. It was written with
spaCy 3.8.16 and `fr_core_news_md` 3.8.0, which `load_texts` checks on every read.

## 2. Examples

**Question:** Do the lemmas keep each description's content words in their base form?

**Method:** Show two training descriptions per class, drawn with a fixed seed, next to their
lemmas.

In [3]:
train.groupby("label").sample(n=2, random_state=42)[["label", "text", "lemmas"]]

,label,text,lemmas
35307,AFF,changement d'affectation d'un local de menuiserie en laboratoire de pâtisserie chocolaterie,changement affectation local menuiserie laboratoire pâtisserie chocolaterie
5752,AFF,transformation d'un dépôt pour création d'un logement,transformation dépôt création logement
39613,AGR,création d'un appartement dans les combles - réfection de la toiture et création de jours divers,création appartement comble réfection toiture création jour
5281,AGR,création d'un appartement dans les combles,création appartement comble
18556,AGV,transformation d'une villa individuelle -création d'une piscine avec terrasse -,transformation villa individuel -création piscine terrasse
17297,AGV,Agrandissement d'une villa et création d'un garage -,agrandissement villa création garage
16177,EQU,aménagements de modération de trafic pour mise en zone 30 km/h,aménagement modération trafic mise zone 30 kilomètre heure
33835,EQU,installation pour téléphonie mobile,installation téléphonie mobile
8949,NOU,Un immeuble de logements - garage souterrain,immeuble logement garage souterrain
4653,NOU,deux immeubles de logements,immeuble logement


**Finding:** The lemmas keep the content words in base form, for example "combles" → "comble" and
"individuelle" → "individuel". Punctuation glued to a word in the source stays attached
(`-création`, `veyrier"remplacement`, `etd'`). Number words disappear with the stop words: "deux
immeubles de logements" becomes "immeuble logement".

## 3. Changed words

**Question:** Which words does lemmatization change most often, and are those lemmas correct?

**Method:** Run the pipeline over the training descriptions again, keep the same tokens as the
stored lemmas, and count the most frequent pairs of lowercased word and lemma that differ.

In [4]:
docs = list(nlp.pipe(train["text"]))
changes = pd.Series(
    [f"{t.lower_} → {t.lemma_}" for doc in docs for t in doc if is_kept(t) and t.lemma_ != t.lower_]
).value_counts()
print(f"{changes.sum():,} changed tokens, {len(changes):,} distinct changes")
changes.head(30)

34,830 changed tokens, 2,678 distinct changes


solaires → solaire                  1475
bureaux → bureau                    1214
intérieures → intérieur             1027
panneaux → panneau                   964
voitures → voiture                   873
transformations → transformation     846
logements → logement                 840
façades → façade                     822
villas → villa                       772
1er → premier                        724
combles → comble                     723
aménagements → aménagement           659
2ème → 2èm                           603
capteurs → capteur                   523
jours → jour                         469
fenêtres → fenêtre                   457
appartements → appartement           436
3ème → 3èm                           429
locaux → local                       427
immeubles → immeuble                 415
4ème → 4èm                           405
garages → garage                     404
places → place                       389
intérieure → intérieur               366
extérieurs → ext

**Finding:** 34,830 training tokens change, across 2,678 distinct word–lemma pairs. The 30 most
frequent changes are correct, mostly plural to singular ("solaires" → "solaire", "villas" →
"villa"), plus "1er" → "premier". Ordinals lose their final letter ("2ème" → "2èm"), but always in
the same way, so no distinction is lost.

## 4. Removed stop words

**Question:** Which stop words are removed, and do any of them carry class information?

**Method:** Count the training tokens that spaCy's French stop-word list removes. A word's lift is
the share of its occurrences in a class divided by that class's share of all training tokens; the
table shows the class with the highest lift. A lift near 1 means the word is spread like the text
itself.

In [5]:
stops = pd.DataFrame(
    [
        (t.lower_, label)
        for doc, label in zip(docs, train["label"], strict=True)
        for t in doc
        if t.is_stop
    ],
    columns=["word", "label"],
)
tokens_per_class = pd.Series(map(len, docs), index=train.index).groupby(train["label"]).sum()
class_share = tokens_per_class / tokens_per_class.sum()
lift = pd.crosstab(stops["word"], stops["label"], normalize="index") / class_share
stop_words = pd.DataFrame(
    {
        "tokens": stops["word"].value_counts(),
        "top_class": lift.idxmax(axis=1),
        "lift": lift.max(axis=1).round(2),
    }
).sort_values("tokens", ascending=False)
print(f"Stop words: {len(stops):,} of {sum(map(len, docs)):,} training tokens")
stop_words.head(30)

Stop words: 86,270 of 232,578 training tokens


,tokens,top_class,lift
word,,,
d',20193,AGV,1.41
de,14529,EQU,1.34
un,9991,AFF,1.32
et,8494,AGV,1.76
une,8081,AGV,2.64
en,4381,AFF,3.85
au,3049,REN,2.12
des,2945,AGR,1.78
pour,2438,EQU,3.34


**Finding:** Stop words are 86,270 of 232,578 training tokens (37%). The most frequent, "d'", "de"
and "un", are spread almost like the text (lift 1.3–1.4), but others are concentrated in one class:
"les" and "dans" in AGR (8.7 and 5.6, mostly from "dans les combles"), "EU" (*eaux usées*, read by
spaCy as the stop word "eu") in EQU (7.5), the number words and "avec" in VIL (2.7–5.6), and "en" in
AFF (3.9). Variant B therefore removes some class information. The list stays unedited;
whether this matters is measured by comparing variants A and B on validation and on the 2013 test.

## 5. Vocabulary and empty texts

**Question:** How much smaller is the vocabulary after preprocessing, and does any description lose
all its words?

**Method:** Split the original and lemmatized training texts with scikit-learn's default word
analyzer, which TF-IDF uses unless configured otherwise, and compare distinct words and words per
text. Then count the rows whose lemmas are empty per partition, and show the training ones.

In [6]:
analyzer = CountVectorizer().build_analyzer()
for column in ["text", "lemmas"]:
    words = train[column].map(analyzer)
    print(
        f"{column}: {len(set().union(*words)):,} distinct words, "
        f"median {words.map(len).median():.0f} per text"
    )

empty = texts["lemmas"].eq("")
display(empty.groupby(texts["partition"]).sum())
train.loc[train["lemmas"].eq(""), ["label", "text"]]

text: 5,393 distinct words, median 6 per text
lemmas: 4,817 distinct words, median 4 per text


partition
test_historical    0
test_recent        0
train_original     0
validation         0
Name: lemmas, dtype: int64

,label,text


**Finding:** With scikit-learn's default analyzer, preprocessing shrinks the training vocabulary
from 5,393 to 4,817 distinct words (11%) and the median description from 6 to 4 words. No
description in any partition loses all its words. The default analyzer drops one-character tokens,
including digits, so the classical models use the token pattern `(?u)\b\w+\b` instead (see
`config.yaml`).